# NDTA 631 — Data Analysis and Visualisation
## Group Assignment: Institutional Effectiveness in South Africa (2014–2018)

**Group members:** _[Full Name — Student Number], ..._
**Module:** NDTA 631 — Data Analysis And Visualization

---

### Datasets
Both datasets are sourced from the **World Bank Data360 / Africa Integrity Indicators** database, filtered to **South Africa (ZAF)**, covering **2014–2018**:

| Code | Indicator | Scale |
|---|---|---|
| `GI_AII_83` | In law, women have equal access to employment opportunities and benefits in the workplace | 0–100 |
| `GI_AII_96` | In practice, roads/bridges networks between towns and cities exist and are maintained | 0–100 |

### The story
Both indicators sit under the same World Bank topic taxonomy: **Prosperity → Institutions → Public Institutions**. One measures legal/regulatory institutional strength, the other measures operational infrastructure-maintenance capacity — both are proxies for overall public-institution effectiveness.

**Research question:** Does South Africa's institutional strength in legal workplace-equality protections move together with its institutional capacity to maintain road/bridge infrastructure over 2014–2018?

**Scope note:** Restricted to South Africa only → n=5 annual observations per indicator. Any correlation is exploratory, not statistically robust — discussed in Section 7.

## 1. Setup — Imports

In [2]:
# Core data handling
import pandas as pd
import numpy as np

# Statistics
from scipy import stats

# Visualisation
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Database
import sqlite3

# Excel export with formatting
import openpyxl
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import LineChart, Reference

# Display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"
DB_PATH = "../db/institutions.db"
EXPORT_DIR = "../exports/"

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Data Preparation (15 marks)

We load both raw World Bank Data360 exports, inspect their structure, check for missing values across the **full source files**, then filter each to South Africa and merge them into one tidy analysis table.

In [7]:
import pandas as pd
import os

RAW_DIR = r"C:\Users\Admin\Downloads\\"

In [8]:
import os

print(os.path.exists(RAW_DIR + "GI_AII_83.csv"))
print(os.path.exists(RAW_DIR + "GI_AII_96.csv"))

True
True


In [9]:
# --- Load raw datasets ---
try:
    women_raw = pd.read_csv(RAW_DIR + "GI_AII_83.csv")
    roads_raw = pd.read_csv(RAW_DIR + "GI_AII_96.csv")

    print(f"GI_AII_83 (women's workplace equality): {women_raw.shape[0]} rows, {women_raw.shape[1]} columns")
    print(f"GI_AII_96 (roads/bridges maintained):    {roads_raw.shape[0]} rows, {roads_raw.shape[1]} columns")

except FileNotFoundError as e:
    raise FileNotFoundError(
        f"Raw data file missing — check the folder. {e}"
    )

GI_AII_83 (women's workplace equality): 270 rows, 41 columns
GI_AII_96 (roads/bridges maintained):    270 rows, 41 columns


In [10]:
# --- Inspect structure ---
women_raw.info()
women_raw[["REF_AREA", "REF_AREA_LABEL", "TIME_PERIOD", "OBS_VALUE", "INDICATOR_LABEL"]].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   STRUCTURE               270 non-null    object 
 1   STRUCTURE_ID            270 non-null    object 
 2   ACTION                  270 non-null    object 
 3   FREQ                    270 non-null    object 
 4   REF_AREA                270 non-null    object 
 5   INDICATOR               270 non-null    object 
 6   SEX                     270 non-null    object 
 7   AGE                     270 non-null    object 
 8   URBANISATION            270 non-null    object 
 9   UNIT_MEASURE            270 non-null    object 
 10  COMP_BREAKDOWN_1        270 non-null    object 
 11  COMP_BREAKDOWN_2        270 non-null    object 
 12  COMP_BREAKDOWN_3        270 non-null    object 
 13  TIME_PERIOD             270 non-null    int64  
 14  OBS_VALUE               270 non-null    fl

,REF_AREA,REF_AREA_LABEL,TIME_PERIOD,OBS_VALUE,INDICATOR_LABEL
0,SEN,Senegal,2018,50.0,"In law, women have equal access to employment ..."
1,DZA,Algeria,2014,100.0,"In law, women have equal access to employment ..."
2,DZA,Algeria,2015,50.0,"In law, women have equal access to employment ..."
3,DZA,Algeria,2016,50.0,"In law, women have equal access to employment ..."
4,DZA,Algeria,2017,50.0,"In law, women have equal access to employment ..."
